In [ ]:
import numpy as np
import pandas as pd
import uuid
import pickle
import torch
from IPython.display import display
from plotnine import *


%load_ext autoreload
%autoreload 2

from vpop_calibration import *

## Define a reference ODE model


In [ ]:
def analytical_model(dose, v, ka, cl, t):
    """Analytical expression of a 1 compartment PK model

    Args:
        t: time in h
        d: Dose in mg
        v: Distribution volume in mL
        ka: Absorption rate constant in mL/h
        cl: Clearance rate constant in mL/h

    Returns:
        y: Predicted concentration
    """
    ke = cl / v
    y1 = dose * ka / (v * (ka - ke)) * (torch.exp(-ke * t) - torch.exp(-ka * t)) + 1e-8
    return y1


variable_names = ["concentration"]

protocol_design = pd.DataFrame({"protocol_arm": ["arm-A", "arm-B"], "dose": [0.1, 10]})
nb_protocols = len(protocol_design)
struct_model = StructuralAnalytical(
    analytical_model, variable_names, protocol_design=protocol_design
)

## Train or load an existing GP surrogate


In [ ]:
model_file = "gp_surrogate_pk_model.pkl"
folder_path = "./"

model_full_path = folder_path + model_file

use_pickle = False
override_existing_pickle = False

In [ ]:
log_nb_patients = 7
param_ranges = {
    "v": {"low": 1.0, "high": 3.0, "log": True},
    "ka": {"low": -2.0, "high": 0.0, "log": True},
    "cl": {"low": -1.0, "high": 1.0, "log": True},
}

t0 = 0
tmax = 5
nb_steps = 20
time_steps = np.linspace(t0, tmax, nb_steps).tolist()
print(f"Simulating {2**log_nb_patients} patients on {nb_protocols} scenario arms")
dataset = generate_training_data(
    struct_model=struct_model,
    log_nb_ind=log_nb_patients,
    ranges=param_ranges,
    time=time_steps,
)
dataset = dataset.loc[dataset.time >= 0.1]
display(dataset)

(
    ggplot(dataset, aes(x="time", y="value", color="id"))
    + theme(legend_position="none")
    + facet_grid(cols="protocol_arm")
    + geom_line()
    + scale_y_continuous(trans="log10")
)

In [ ]:
if (override_existing_pickle) or (not use_pickle):
    learned_ode_params = list(param_ranges.keys())
    descriptors = learned_ode_params + ["time"]

    # Instantiate a GP
    myGP = GP(
        training_df=dataset,
        descriptors=descriptors,
        deep_kernel=True,
        nb_inducing_points=100,
        nb_training_iter=200,
        training_proportion=0.7,
        learning_rate=0.05,
        lr_decay=0.99,
        nb_features=10,
        min_delta=0.001,
        log_inputs=learned_ode_params,
        log_outputs=variable_names,
    )
    # Train the GP
    myGP.train()

    if use_pickle and override_existing_pickle:
        with open(model_full_path, "wb") as file:
            pickle.dump(myGP, file)
        print(f"Model saved to {model_file}")
elif use_pickle and (not override_existing_pickle):
    try:
        with open(model_full_path, "rb") as file:
            myGP = pickle.load(file)

        print("Model loaded successfully!")

    except FileNotFoundError:
        print(
            f"File not found. Please make sure '{model_full_path}' exists and is in the correct directory."
        )

In [ ]:
myGP.plot_all_solutions("training")
myGP.plot_obs_vs_predicted("training")

## Generate a synthetic data set using a NLME from the ODEs directly


In [ ]:
nlme_ground_truth = {
    "pdu": {
        "ka": {
            "prior": 0.4,
            "prior_omega": 0.1,
        },
        "cl": {"prior": 1.0, "prior_omega": 0.1},
        "v": {"prior": 8.0, "prior_omega": 0.1},
    },
    "error_model": {"concentration": {"error_type": "proportional", "sigma": 0.025}},
}


# Create a patient data frame
# It should contain at the very minimum one `id` per patient
nb_patients = 15
obs_df = generate_synthetic_data(
    struct_model=struct_model,
    param_distrib=nlme_ground_truth,
    nb_patients=nb_patients,
    time=time_steps,
)
obs_df = obs_df.loc[obs_df.time > 0.1]


(
    ggplot(obs_df, aes(x="time", y="value", color="id"))
    + theme(legend_position="none")
    + facet_grid(cols="protocol_arm")
    + scale_y_continuous(trans="log10")
    + geom_line()
)

## Optimize the GP surrogate using SAEM


In [ ]:
# Create a structural model
structural_gp = StructuralGp(myGP)

print(structural_gp.training_ranges)
nlme_prior = {
    "pdu": {
        "ka": {
            "prior": 0.5,
            "prior_omega": 0.25,
            "constraint": structural_gp.training_ranges["ka"],
        },
        "cl": {
            "prior": 1.0,
            "prior_omega": 0.25,
            "constraint": structural_gp.training_ranges["cl"],
        },
        "v": {
            "prior": 10.0,
            "prior_omega": 0.25,
            "constraint": structural_gp.training_ranges["v"],
        },
    },
    "error_model": {"concentration": {"error_type": "proportional", "sigma": 0.5}},
}
config_saem = SaemConfigDict(
    nb_phase1_iterations=200,
    nb_phase2_iterations=200,
    mcmc_first_burn_in=20,
    mcmc_nb_transitions=1,
)
# Create a NLME model
nlme_surrogate = NlmeModel(
    structural_model=structural_gp,
    prior_params=nlme_prior,
    df=obs_df,
    config=Config(saem=config_saem),
)

In [ ]:
nlme_surrogate.optimizer.run()

In [ ]:
nlme_surrogate.diagnostics.sample_conditional_distribution(300)

In [ ]:
nlme_surrogate.plot.map_estimates()

In [ ]:
nlme_surrogate.plot.all_individual_map_estimates(1, 3, 3, randomize=True)

In [ ]:
nlme_surrogate.plot.map_estimates_gof(5, 5)

In [ ]:
warnings, ranges = nlme_surrogate.plot.check_surrogate_validity_gp(5, 4)

In [ ]:
nlme_surrogate.plot.weighted_residuals("iwres")
# plot_weighted_residuals(nlme_surrogate, "pwres")
# plot_weighted_residuals(nlme_surrogate, "npde")

In [ ]:
nlme_surrogate.plot.map_vs_posterior(4)